# System Results — Sound Recognition & Speech-to-Text
**Paper:** *A Non-Prosthetic Assistive System for Persons with Hearing Losses: Design and Experimental Investigation*  
F. AlHayek · R. Alsubaiei · M. Alsahhaf · G. Alajmi · A. Almutairi · K. Youssef · S. Said · S. Alkork  
American University of the Middle East

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.dpi'       : 120,
    'savefig.dpi'      : 150,
    'savefig.bbox'     : 'tight',
})

C_HEADER = '#1B2A4A'
C_BEST   = '#D5F0DD'
C_ROW_A  = '#FFFFFF'
C_ROW_B  = '#F4F6F9'
C_BLUE   = '#2E86C1'
C_GREEN  = '#27AE60'
C_ORANGE = '#E67E22'
C_RED    = '#C0392B'
C_PURPLE = '#8E44AD'

REPO_ROOT = os.path.dirname(os.getcwd())
PLOTS_DIR = os.path.join(REPO_ROOT, '4_results', 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

def draw_table(ax, headers, rows, col_widths=None, best_row=None,
               row_colors=None, header_color=C_HEADER, fontsize=10.5):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    n_rows = len(rows); n_cols = len(headers)
    if col_widths is None: col_widths = [1/n_cols]*n_cols
    row_h = 0.82/(n_rows+1); y_start = 0.95
    xs = [0.02]
    for w in col_widths[:-1]: xs.append(xs[-1]+w*0.96)
    def cell(ax, x, y, w, h, text, fc, tc='black', fw='normal', fs=fontsize):
        ax.add_patch(plt.Rectangle((x,y),w*0.96,h*0.88,facecolor=fc,
                     edgecolor='white',linewidth=1.2,transform=ax.transAxes,clip_on=False))
        ax.text(x+w*0.48,y+h*0.44,text,ha='center',va='center',fontsize=fs,
                color=tc,fontweight=fw,transform=ax.transAxes)
    for h_txt,w,x in zip(headers,col_widths,xs):
        cell(ax,x,y_start-row_h,w,row_h,h_txt,fc=header_color,tc='white',fw='bold')
    for i,row in enumerate(rows):
        y = y_start-row_h*(i+2)
        if row_colors and i<len(row_colors): bg=row_colors[i]
        elif best_row is not None and i==best_row: bg=C_BEST
        else: bg=C_ROW_A if i%2==0 else C_ROW_B
        for val,w,x in zip(row,col_widths,xs):
            fw='bold' if (best_row is not None and i==best_row) else 'normal'
            cell(ax,x,y,w,row_h,str(val),fc=bg,fw=fw)

print('Setup complete.')

---
## Fig. 1 — Table IV: Sound Recognition (YAMNet)

In [ ]:
SR_CATS = ['Speech', 'Alarms', 'Animals', 'Tools', 'Footsteps']
SR_ACC  = [100, 95, 92, 90, 88]
SR_NOTE = [
    'Confidence 0.97–1.0; silence → 1.0',
    'Siren, fire alarm — critical for haptic alerts',
    'Hierarchical: animal → cat → meow',
    'Drawer, pans, hand tools',
    'Confidence drops at distance > 4 m',
]

fig = plt.figure(figsize=(15, 8))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# ── (a) Table ─────────────────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0])
draw_table(
    ax_t,
    headers    = ['Sound Category', 'Accuracy', 'Observation'],
    col_widths = [0.24, 0.16, 0.56],
    rows       = [[c, f'{a}%', n] for c, a, n in zip(SR_CATS, SR_ACC, SR_NOTE)],
    best_row   = 0,
    fontsize   = 9.5,
)
ax_t.set_title('(a) Table IV — YAMNet Sound Recognition\nby Category (Indoor, ~6×5×2.8 m)',
               fontweight='bold', pad=6)

# ── (b) Horizontal bar chart ──────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1])
bar_colors = [C_GREEN if a==100 else C_BLUE if a>=92 else C_ORANGE for a in SR_ACC]
bars = ax_b.barh(SR_CATS[::-1], SR_ACC[::-1], color=bar_colors[::-1],
                 edgecolor='white', height=0.55)
for bar, val in zip(bars, SR_ACC[::-1]):
    ax_b.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
              f'{val}%', va='center', fontsize=12, fontweight='bold')
ax_b.axvline(90, color='#AAAAAA', linestyle='--', lw=1.3, label='90% line')
ax_b.set_xlim(0, 112)
ax_b.set_xlabel('Recognition Accuracy (%)')
ax_b.set_title('(b) Accuracy by Sound Category', fontweight='bold')
ax_b.legend(fontsize=9)
ax_b.grid(axis='x', alpha=0.3)

avg = np.mean(SR_ACC)
ax_b.axvline(avg, color=C_RED, linestyle=':', lw=1.5, label=f'Average = {avg:.0f}%')
ax_b.legend(fontsize=9)

fig.suptitle('Fig. 1 — YAMNet Sound Recognition Performance',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'sys_fig1_sound_recognition.png'))
plt.show()

---
## Fig. 2 — Table V: STT System Comparison

In [ ]:
fig = plt.figure(figsize=(15, 8))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38)

# ── (a) Comparison table ──────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0])
draw_table(
    ax_t,
    headers    = ['Metric', 'Faster-Whisper', 'Google STT'],
    col_widths = [0.38, 0.29, 0.29],
    rows = [
        ['Overall Accuracy',       '37.5%',    '87.74%'],
        ['Noise Robustness',       'Limited',  'High'],
        ['Kuwaiti Dialect Support','Moderate', 'Good'],
        ['Transcription Latency',  '3.5–7 s',  '4–8 s'],
        ['Deployment',             'On-device','Cloud streaming'],
    ],
    best_row   = 0,
    row_colors = [C_BEST, C_ROW_B, C_ROW_A, C_ROW_B, C_ROW_A],
    fontsize   = 10,
)
ax_t.set_title('(a) Table V — Faster-Whisper vs. Google STT',
               fontweight='bold', pad=6)

# ── (b) Accuracy + latency visual ────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1])
systems  = ['Faster-Whisper', 'Google STT']
acc_vals = [37.5, 87.74]
lat_mid  = [5.25, 6.0]
lat_err  = [1.75, 2.0]

clrs = [C_RED, C_GREEN]
bars = ax_b.bar(systems, acc_vals, color=clrs, edgecolor='white', width=0.45)
for bar, val in zip(bars, acc_vals):
    ax_b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.6,
              f'{val}%', ha='center', fontsize=13, fontweight='bold')

ax_b2 = ax_b.twinx()
ax_b2.errorbar(systems, lat_mid, yerr=lat_err, fmt='D', color='#333333',
               capsize=8, capthick=2, markersize=8, linewidth=2, label='Latency (s)')
ax_b2.set_ylabel('Latency (s)')
ax_b2.set_ylim(0, 12)
ax_b2.legend(loc='upper right', fontsize=9)

ax_b.set_ylabel('Overall Accuracy (%)')
ax_b.set_ylim(0, 110)
ax_b.set_title('(b) Accuracy & Latency Comparison', fontweight='bold')
ax_b.grid(axis='y', alpha=0.3)

fig.suptitle('Fig. 2 — STT System Comparison: Faster-Whisper vs. Google Cloud STT',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'sys_fig2_stt_comparison.png'))
plt.show()

---
## Fig. 3 — Table VI: Arabic STT Evaluation

In [ ]:
AR_SENTS  = ['S1: السلام عليكم ورحمة الله وبركاته',
             'S2: شلونكم شخباركم عساكم بخير',
             'S3: الاحد إن شاء الله تقديم مشروع التخرج',
             'S4: بعدها عندي امتحان واحد',
             'S5: يوم التخرج تاريخ 5/18 بإذن الله']
AR_LAT_0  = [4.36, 4.58, 6.59, 6.74, 8.66]
AR_LAT_2  = [4.50, 4.61, 6.73, 6.32, 6.60]
AR_LAT_5  = [4.20, 4.39, 6.64, 6.74, 8.01]
AR_ERRORS = ['None', '1 word (5 m)', '1 word (0 m, 5 m)', '1 word (5 m)', 'None']

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.40)

# ── (a) Per-sentence table ────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0])
draw_table(
    ax_t,
    headers    = ['Sentence', 'Lat. 0 m (s)', 'Lat. 2 m (s)', 'Lat. 5 m (s)', 'Errors'],
    col_widths = [0.46, 0.13, 0.13, 0.13, 0.13],
    rows = [[s, f'{l0:.2f}', f'{l2:.2f}', f'{l5:.2f}', e]
            for s, l0, l2, l5, e in zip(AR_SENTS, AR_LAT_0, AR_LAT_2, AR_LAT_5, AR_ERRORS)],
    row_colors = [C_BEST if e=='None' else C_ROW_A if i%2==0 else C_ROW_B
                  for i, e in enumerate(AR_ERRORS)],
    fontsize   = 9.5,
)
ax_t.set_title('(a) Table VI — Arabic STT Evaluation  (WER = 15.38%)',
               fontweight='bold', pad=6)

# ── (b) Latency vs distance ───────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1])
distances = [0, 2, 5]
sent_colors = [C_BLUE, C_GREEN, C_ORANGE, C_PURPLE, C_RED]
for i, (lats, clr) in enumerate(zip([AR_LAT_0, AR_LAT_2, AR_LAT_5], [None]*3)):
    pass
for s_idx, (lat_set, clr) in enumerate(zip(
        [[AR_LAT_0[i], AR_LAT_2[i], AR_LAT_5[i]] for i in range(5)],
        sent_colors)):
    ax_b.plot(distances, lat_set, 'o-', color=clr, lw=2, markersize=7,
              label=f'S{s_idx+1}')

ax_b.set_xlabel('Distance from Microphone (m)')
ax_b.set_ylabel('Transcription Latency (s)')
ax_b.set_xticks(distances)
ax_b.set_ylim(2, 11)
ax_b.set_title('(b) Latency vs. Distance — Arabic Sentences', fontweight='bold')
ax_b.legend(fontsize=9, ncol=5, loc='upper center')
ax_b.grid(True, alpha=0.3)

# Avg lines
for dist_idx, (lats, d) in enumerate(zip([AR_LAT_0, AR_LAT_2, AR_LAT_5], distances)):
    avg = np.mean(lats)
    ax_b.annotate(f'avg\n{avg:.1f}s', xy=(d, avg), xytext=(d+0.08, avg+0.35),
                  fontsize=8, color='#555555',
                  arrowprops=dict(arrowstyle='->', color='#999999', lw=0.8))

fig.suptitle('Fig. 3 — Arabic STT Evaluation (Google Cloud STT)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'sys_fig3_arabic_stt.png'))
plt.show()

---
## Fig. 4 — Table VII: English STT Evaluation

In [ ]:
EN_SENTS  = ['S1: hello how are you',
             'S2: I hope you are doing well',
             'S3: this test for graduation project',
             'S4: we have 3 features one of this localization',
             'S5: sound recognition and speech to text']
EN_LAT_0  = [4.70, 5.62, 7.23, 6.81, 7.67]
EN_LAT_25 = [4.45, 6.55, 6.71, 6.97, 7.09]
EN_LAT_5  = [4.03, 4.76, 4.52, 8.54, 6.43]
EN_ERRORS = ['None', 'None', '1 word (0 m)', 'None', 'None']

fig = plt.figure(figsize=(15, 9))
gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.40)

# ── (a) Table ─────────────────────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0])
draw_table(
    ax_t,
    headers    = ['Sentence', 'Lat. 0 m (s)', 'Lat. 2.5 m (s)', 'Lat. 5 m (s)', 'Errors'],
    col_widths = [0.46, 0.13, 0.14, 0.13, 0.12],
    rows = [[s, f'{l0:.2f}', f'{l25:.2f}', f'{l5:.2f}', e]
            for s, l0, l25, l5, e in zip(EN_SENTS, EN_LAT_0, EN_LAT_25, EN_LAT_5, EN_ERRORS)],
    row_colors = [C_BEST if e=='None' else C_ROW_B
                  for e in EN_ERRORS],
    fontsize   = 9.5,
)
ax_t.set_title('(a) Table VII — English STT Evaluation  (WER = 3.70%)',
               fontweight='bold', pad=6)

# ── (b) Latency vs distance ───────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1])
distances_en = [0, 2.5, 5]
sent_colors  = [C_BLUE, C_GREEN, C_ORANGE, C_PURPLE, C_RED]
for s_idx, (lat_set, clr) in enumerate(zip(
        [[EN_LAT_0[i], EN_LAT_25[i], EN_LAT_5[i]] for i in range(5)],
        sent_colors)):
    ax_b.plot(distances_en, lat_set, 'o-', color=clr, lw=2, markersize=7,
              label=f'S{s_idx+1}')

ax_b.set_xlabel('Distance from Microphone (m)')
ax_b.set_ylabel('Transcription Latency (s)')
ax_b.set_xticks(distances_en)
ax_b.set_ylim(2, 11)
ax_b.set_title('(b) Latency vs. Distance — English Sentences', fontweight='bold')
ax_b.legend(fontsize=9, ncol=5, loc='upper center')
ax_b.grid(True, alpha=0.3)

fig.suptitle('Fig. 4 — English STT Evaluation (Google Cloud STT)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'sys_fig4_english_stt.png'))
plt.show()

---
## Fig. 5 — WER & Latency Analysis

In [ ]:
fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ── (a) WER comparison ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
langs     = ['Arabic\n(incl. Kuwaiti\ndialect)', 'English']
wer_vals  = [15.38, 3.70]
wer_clrs  = [C_ORANGE, C_GREEN]
bars1 = ax1.bar(langs, wer_vals, color=wer_clrs, edgecolor='white', width=0.45)
for bar, val in zip(bars1, wer_vals):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{val}%', ha='center', fontsize=12, fontweight='bold')
ax1.set_ylabel('Word Error Rate (%)')
ax1.set_ylim(0, 22)
ax1.set_title('(a) WER by Language', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# ── (b) Mean latency vs distance (AR vs EN) ───────────────────────────────────
ax2 = fig.add_subplot(gs[1])
dist_ar = [0, 2, 5]
dist_en = [0, 2.5, 5]
mean_ar = [np.mean([AR_LAT_0[i]  for i in range(5)]),
           np.mean([AR_LAT_2[i]  for i in range(5)]),
           np.mean([AR_LAT_5[i]  for i in range(5)])]
mean_en = [np.mean([EN_LAT_0[i]  for i in range(5)]),
           np.mean([EN_LAT_25[i] for i in range(5)]),
           np.mean([EN_LAT_5[i]  for i in range(5)])]
ax2.plot(dist_ar, mean_ar, 'o-', color=C_ORANGE, lw=2.5, markersize=9, label='Arabic')
ax2.plot(dist_en, mean_en, 's-', color=C_GREEN,  lw=2.5, markersize=9, label='English')
for d, v in zip(dist_ar, mean_ar):
    ax2.text(d+0.05, v+0.12, f'{v:.1f}s', fontsize=9, color=C_ORANGE)
for d, v in zip(dist_en, mean_en):
    ax2.text(d+0.05, v-0.28, f'{v:.1f}s', fontsize=9, color=C_GREEN)
ax2.set_xlabel('Distance (m)')
ax2.set_ylabel('Mean Latency (s)')
ax2.set_ylim(3, 9)
ax2.set_title('(b) Mean Latency vs. Distance', fontweight='bold')
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

# ── (c) Accuracy summary ──────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
summary_labels = ['Arabic\nAccuracy', 'English\nAccuracy']
summary_acc    = [100-15.38, 100-3.70]
summary_clrs   = [C_ORANGE, C_GREEN]
bars3 = ax3.bar(summary_labels, summary_acc, color=summary_clrs, edgecolor='white', width=0.45)
for bar, val in zip(bars3, summary_acc):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax3.set_ylabel('Accuracy  (1 − WER) (%)')
ax3.set_ylim(0, 110)
ax3.set_title('(c) Transcription Accuracy', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

fig.suptitle('Fig. 5 — STT WER and Latency Analysis',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'sys_fig5_stt_analysis.png'))
plt.show()

---
## Fig. 6 — Full System Performance Overview

In [ ]:
fig = plt.figure(figsize=(15, 8))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38)

# ── (a) System summary table ──────────────────────────────────────────────────
ax_t = fig.add_subplot(gs[0])
draw_table(
    ax_t,
    headers    = ['Module', 'Key Result', 'Input Channel'],
    col_widths = [0.26, 0.43, 0.27],
    rows = [
        ['Sound\nLocalization',   '99.22% offline\n90.7% ±15° real-time\nMAE 0.4° / 9.5°', 'Ch 2–5\n(raw mics)'],
        ['Sound\nRecognition',    'Avg 93%\n(90–100% per category)', 'Ch 1\n(ASR audio)'],
        ['Speech-to-Text\n(STT)', 'WER 3.7% EN\nWER 15.4% AR\nLatency 4–8 s',  'Ch 0\n(Conference)'],
    ],
    fontsize = 9.5,
)
ax_t.set_title('(a) Full System — Module Summary',
               fontweight='bold', pad=6)

# ── (b) Radar chart ───────────────────────────────────────────────────────────
ax_r = fig.add_subplot(gs[1], polar=True)
radar_labels = ['Localization\n(Offline)', 'Localization\n(Real-Time ±15°)',
                'Sound Recog.\n(avg)', 'STT English\n(accuracy)', 'STT Arabic\n(accuracy)']
radar_vals   = [99.22, 90.7, 93.0, 96.3, 84.62]
N = len(radar_labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
vals_p = radar_vals + [radar_vals[0]]
angs_p = angles    + [angles[0]]

ax_r.plot(angs_p, vals_p, 'o-', lw=2.5, color=C_BLUE)
ax_r.fill(angs_p, vals_p, alpha=0.22, color=C_BLUE)
ax_r.set_xticks(angles)
ax_r.set_xticklabels(radar_labels, fontsize=9)
ax_r.set_ylim(0, 110)
ax_r.set_yticks([25, 50, 75, 90, 100])
ax_r.set_yticklabels(['25', '50', '75', '90', '100'], fontsize=8, color='#777777')
ax_r.set_title('(b) System Performance Radar', fontweight='bold', pad=20)
ax_r.grid(True, alpha=0.4)

for angle, val in zip(angles, radar_vals):
    ax_r.annotate(f'{val:.1f}%',
                  xy=(angle, val), xytext=(0, 8), textcoords='offset points',
                  fontsize=9, fontweight='bold', ha='center', color=C_BLUE)

fig.suptitle('Fig. 6 — Integrated System Performance Overview',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'sys_fig6_system_overview.png'))
plt.show()
print('All figures saved to:', PLOTS_DIR)